## Webscraping - practice

In [ ]:
from IPython.display import display
import json
import pandas as pd
pd.options.display.max_columns = None # Display all columns of a dataframe
pd.options.display.max_rows = 700
from pprint import pprint
import re
import requests
from bs4 import BeautifulSoup

### Static elements 

In [ ]:
url = "https://fr.wikipedia.org/wiki/Liste_des_communes_de_la_R%C3%A9gion_wallonne"
# Wikipedia (like many websites) refuses requests that do not identify themselves:
# without a "User-Agent" header the server answers 403 Forbidden.
headers = {"User-Agent": "Mozilla/5.0 (ECON2206 student project)"}
response = requests.get(url, headers=headers)
print(response.status_code)   # 200 = OK. Always check this before parsing!
html = response.text
soup = BeautifulSoup(html, 'html.parser')

In [ ]:
print(soup.prettify()[:1000])

> 📝 **Note for students** — before writing any scraping code, open the page in your browser, right-click on the element you want → **Inspect**. This shows the HTML tags (`<table>`, `<tr>` = row, `<td>` = cell, ...) and their attributes (`class`, `id`) that you can use to locate the data with `find()` / `find_all()`.

#### Task 1

Get the name of all municipalities of Wallonia

In [ ]:
# The municipalities are stored in a <table> with the class "wikitable"
table = soup.find('table', class_='wikitable')

# Each row of the table is a <tr>. The first row contains the column headers (<th>), so we skip it.
rows = table.find_all('tr')[1:]

# In each row, the first cell <td> contains the French name of the municipality
municipalities = [row.find_all('td')[0].get_text(strip=True) for row in rows]

print(len(municipalities))   # 261 municipalities in Wallonia
municipalities[:10]

> 📝 **Note for students** — `find()` returns the **first** matching element, `find_all()` returns a **list** of all matching elements. `get_text(strip=True)` extracts the visible text and removes surrounding white spaces. Note the underscore in `class_`: `class` is a reserved word in Python.

#### Task 2

Get the population of each municipality

In [ ]:
# Let's look at the headers to find the position of the population column
headers_row = [th.get_text(strip=True) for th in table.find('tr').find_all('th')]
print(headers_row)   # 'Population (2020)' is the 6th column -> index 5

In [ ]:
def clean_number(cell):
    """Convert a table cell such as '10 736' into the integer 10736."""
    # Wikipedia adds invisible characters (used to sort the table) inside <s style="visibility:hidden">
    # tags. We remove them first, otherwise they would end up in our number.
    for hidden in cell.find_all(style=re.compile(r'visibility:\s*hidden')):
        hidden.decompose()
    text = cell.get_text(strip=True)
    # Keep only the digits (removes the non-breaking spaces used as thousands separators, commas, ...)
    digits = re.sub(r'[^\d]', '', text)
    return int(digits) if digits else None

populations = [clean_number(row.find_all('td')[5]) for row in rows]
populations[:10]

> 📝 **Note for students** — scraped data is **always text**. You have to clean it (remove spaces, separators, footnotes like `[1]`) and convert it to numbers yourself. Regular expressions (`re` module) are very handy for that: `[^\d]` means *any character that is not a digit*.

#### Task 3

Send the data in a appropriate dataframe

In [ ]:
df_communes = pd.DataFrame({'municipality': municipalities, 'population': populations})

# Sanity checks: the number of rows, the type of the columns, and the total population (~3.6 million)
print(df_communes.shape)
print(df_communes.dtypes)
print("Total population of Wallonia:", df_communes['population'].sum())

df_communes.sort_values('population', ascending=False).head(10)

> 📝 **Note for students** — always **check your scraped data**: number of rows, missing values, and an order of magnitude you know (Wallonia has about 3.6 million inhabitants). For simple HTML tables, `pd.read_html()` can also parse every `<table>` of a page directly into a list of DataFrames — but you still need to clean the columns.

### Dynamic elements

Let's scrape data from https://fr.be.getaround.com/ !

> 📝 **Note for students** — **static vs dynamic pages** — `requests` only downloads the initial HTML sent by the server. On dynamic websites such as Getaround, the content (car list, prices) is generated **by JavaScript in the browser** after the page loads, so it is not in `response.text`. We need a *simulated browser* with **Selenium** (see notebook 2.3). The selectors below are examples: websites change their HTML often, so inspect the page and adapt them.

#### Task 1

Search any location you want and get the price of the first car. 

In [ ]:
# Selenium 4 syntax: the driver is downloaded automatically (Selenium Manager), no chromedriver path needed
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
location = "Liège"
driver.get(f"https://fr.be.getaround.com/search?address={location}")

# Wait until the elements are loaded (better than a fixed time.sleep):
# here we wait at most 10 seconds for the first price to appear on the page.
wait = WebDriverWait(driver, 10)

# Accept cookies if the banner is displayed (otherwise it blocks the clicks)
try:
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Accepter')]"))).click()
except Exception:
    pass  # no cookie banner

# Inspect the page to find the tag/class of the price; adapt the selector if the website changed
first_price = wait.until(EC.presence_of_element_located((By.XPATH, "//*[contains(text(), '€')]")))
print("Price of the first car:", first_price.text)

> 📝 **Note for students** — XPath cheat sheet: `//tag` any `tag` in the page, `[@class='x']` with attribute `class` equal to `x`, `[contains(text(), '€')]` whose text contains `€`. In Chrome, *Inspect → right-click on the element → Copy → Copy XPath* gives an XPath, but such absolute paths (`/html/body/div[1]/...`) break as soon as the page layout changes.

#### Task 2

Loop through the car list of a location and get the price of all cars. 

In [ ]:
# find_elements (with an "s") returns a LIST of all the matching elements
time.sleep(2)  # leave time for all the cards to load
price_elements = driver.find_elements(By.XPATH, "//*[contains(text(), '€')]")

prices = []
for element in price_elements:
    text = element.text
    digits = re.sub(r'[^\d,\.]', '', text).replace(',', '.')   # '45,50 €' -> '45.50'
    if digits:
        prices.append(float(digits))

print(len(prices), "prices found")
print(prices)

# Results usually come page by page: to get more cars, scroll down or click on the "next page" button,
# then repeat the extraction in a loop (while a next button exists).
driver.quit()   # ALWAYS close the browser at the end

> 📝 **Note for students** — be a responsible scraper: check the website's terms of use and `robots.txt`, add pauses (`time.sleep`) between requests so you do not overload the server, and never scrape personal data.